# **5G Deployment Example with OCUDU ZMQ (1)**

In this example, we will deploy a 5G network with:

- Open5GS core with: AMF, AUSF, BSF, NSSF, NRF, PCF, SMF, UPF, UDM, UDR
- OCUDU CU-CP, CU-UP, and DU with ZeroMQ
- SRSUE with ZeroMQ as simulated UE

Reference: https://docs.ocudu.org/tutorials/srsue/#zeromq-based-setup

First, we install the required dependencies:

## Install Dependencies:

Install the `ifran` library:

In [ ]:
! pip install ifran

Install the Ansible dependencies:

In [ ]:
! ansible-galaxy install -r ../../ansible/requirements.yaml

Install the SLICES CLI used for the BI to create VMs:

In [ ]:
! pip install slices-cli --extra-index-url=https://doc.slices-ri.eu/pypi/

Authenticate with your SLICES account:

In [ ]:
! slices auth login

Make sure you have an SSH key that you add to SLICES using the command:

```bash
slices pubkey register /path/to/ssh/key
```

which will be use to grant SSH access to the provisioned SLICES machines.

## Input 5G Network Description:

In [1]:
from ifran import MetaModel

metamodel = MetaModel.from_yaml(
"""
rus:
  sim_ru_du:
    txGainDB: 0
    rxGainDB: 0

  sim_ru_ue:
    txGainDB: 50
    rxGainDB: 40

ran:
  components:
    cucp:
      type: cu_cp
    cuup:
      type: cu_up
      config:
        cu_cp: cucp
    du:
      type: du
      config:
        ru: sim_ru_du
        cu_cp: cucp
        cu_up: cuup 
        radioConfiguration:
          nrBand: 3
          downlinkArfcn: 368500
          channelBandwidthMhz: 10
          subcarrierSpacingKhz: 15
          ssbPeriodicityMs: 10
          prachConfigurationIndex: 1

ues:
  sim_ue:
    config:
      nrBands: [3]
      ru: sim_ru_ue
      gnbs: [du]

slices:
  internet:
    sst: 1
    sd: 0xFFFFFF # explicitly set sd to avoid core regection
"""
)

In [2]:
from ifran import DeploymentModel

deployment_model = DeploymentModel.from_yaml(
"""
params:
  provisioning:
    project: post5g-beta
    experiment: ifran
    flavor: xlarge # ocudu zmq needs more than 5 CPU cores (see: https://github.com/srsran/srsRAN_Project/issues/1472)

ran:
  implementation: ocudu
  components:
    cucp:
      type: cu_cp
      advanced_config:
        inactivity_timer: 7200
    du:
      type: du
      advanced_config:
        srate: 11.52
        base_srate: "11.52e6"
        pdcch:
          common:
            ss0_index: 0
            coreset0_index: 6
          dedicated:
            ss2_type: common
            dci_format_0_1_and_1_1: false
        pdsch:
          mcs_table: qam64
        pusch:
          mcs_table: qam64

ues:
  sim_ue:
    type: containerized
    implementation: srsue
    advanced_config:
      srate: "11.52e6"
"""
)

## Provisioning:

In [3]:
from ifran import ProvisioningManagement

prov = ProvisioningManagement(metamodel=metamodel, deployment_model=deployment_model)

In [4]:
await prov.provision()

[WARNING]: No inventory was parsed, only implicit localhost is available
[WARNING]: No inventory was parsed, only implicit localhost is available
[WARNING]: No inventory was parsed, only implicit localhost is available
[WARNING]: No inventory was parsed, only implicit localhost is available
[WARNING]: provided hosts list is empty, only localhost is available. Note that the implicit localhost does not match 'all'
[WARNING]: provided hosts list is empty, only localhost is available. Note that the implicit localhost does not match 'all'
[WARNING]: provided hosts list is empty, only localhost is available. Note that the implicit localhost does not match 'all'
[WARNING]: provided hosts list is empty, only localhost is available. Note that the implicit localhost does not match 'all'

PLAY [localhost] ***************************************************************

PLAY [localhost] ***************************************************************

PLAY [localhost] ******************************

## Kubernetes Deployment:

In [5]:
from ifran import K8SManagement

k8s = K8SManagement(prov)

In [6]:
await k8s.deploy()


PLAY [node1] *******************************************************************

PLAY [node3] *******************************************************************

TASK [k8s/prepare : Set time zone to 'Europe/Paris' on 'node1'] ****************

PLAY [node2] *******************************************************************

TASK [k8s/prepare : Set time zone to 'Europe/Paris' on 'node3'] ****************

TASK [k8s/prepare : Set time zone to 'Europe/Paris' on 'node2'] ****************

PLAY [node4] *******************************************************************

TASK [k8s/prepare : Set time zone to 'Europe/Paris' on 'node4'] ****************
changed: [node3]

TASK [k8s/prepare : Install base packages required for Kubernetes on 'node3'] ***
changed: [node4]

TASK [k8s/prepare : Install base packages required for Kubernetes on 'node4'] ***
changed: [node1]

TASK [k8s/prepare : Install base packages required for Kubernetes on 'node1'] ***
changed: [node2]

TASK [k8s/prepare : Instal

## 5G Core Deployment:

In [7]:
from ifran import CoreManagement

core = CoreManagement(k8s)

In [8]:
await core.deploy()


PLAY [node1] *******************************************************************

TASK [helm/prepare : Create Helm build directory for 'open5gs-database' chart] ***

PLAY [node1] *******************************************************************

TASK [helm/prepare : Create Helm build directory for 'open5gs-amf' chart] ******

PLAY [node1] *******************************************************************

TASK [helm/prepare : Create Helm build directory for 'open5gs-ausf' chart] *****

PLAY [node1] *******************************************************************

TASK [helm/prepare : Create Helm build directory for 'open5gs-nrf' chart] ******

PLAY [node1] *******************************************************************

TASK [helm/prepare : Create Helm build directory for 'open5gs-nssf' chart] *****

PLAY [node1] *******************************************************************

PLAY [node1] *******************************************************************

TASK [helm/pr

## RAN Deployment:

In [9]:
from ifran import RANManagement

ran = RANManagement(core)

In [10]:
await ran.deploy()


PLAY [node1] *******************************************************************

TASK [helm/prepare : Create Helm build directory for 'ocudu-cucp' chart] *******

PLAY [node1] *******************************************************************

PLAY [node1] *******************************************************************

TASK [helm/prepare : Create Helm build directory for 'ocudu-du' chart] *********
TASK [helm/prepare : Create Helm build directory for 'ocudu-cuup' chart] *******

changed: [node1]

TASK [helm/prepare : Compress custom Helm chart directory for 'ocudu-cucp' if provided on localhost] ***
skipping: [node1]

TASK [helm/prepare : Compress built-in Helm chart directory for 'ocudu-cucp' on localhost] ***
changed: [node1 -> localhost]

TASK [helm/prepare : Extract Helm chart archive for 'ocudu-cucp' from localhost] ***
changed: [node1]
changed: [node1]

TASK [helm/prepare : Compress custom Helm chart directory for 'ocudu-cuup' if provided on localhost] ***
TASK [helm/prep

Get OCUDU ZMQ DU logs:

In [11]:
print(await ran.get_logs(components=["du"], quiet=True))


--== OCUDU DU (commit 050a2bb) ==--

2026-11-27T19:44:43.791708 [GNB     ] [I] Built in Release mode using commit 050a2bb on branch HEAD
2026-11-27T19:44:43.792615 [CONFIG  ] [I] Input configuration (only non-default values): 
dryrun: false
gnb_id: 1
gnb_id_bit_length: 32
gnb_du_id: 1
log:
  filename: stdout
  all_level: info
  config_level: info
metrics:
  autostart_stdout_metrics: true
f1ap:
  addrs: 10.244.167.11
  bind_addrs: 10.244.137.230
f1u:
  bind_port: 2153
  peer_port: 2153
  socket: "bind_addr: 10.244.137.230"
cell_cfg:
  pci: 0
  dl_arfcn: 368500
  band: 3
  common_scs: 15
  channel_bandwidth_MHz: 10
  nof_antennas_ul: 1
  nof_antennas_dl: 1
  plmn: 00101
  tac: 1
  slicing: "sst: 1\nsd: 16777215"
  ssb:
    ssb_period: 10
  pdcch:
    common:
      coreset0_index: 6
      ss0_index: 0
    dedicated:
      dci_format_0_1_and_1_1: false
      ss2_type: common
  pdsch:
    mcs_table: qam64
  pusch:
    mcs_table: qam64
  prach:
    prach_config_index: 1
ru_sdr:
  srate: 11.

## UE Deployement:

In [12]:
from ifran import UEManagement

ue = UEManagement(ran)

In [13]:
await ue.deploy()


PLAY [node1] *******************************************************************

TASK [helm/prepare : Create Helm build directory for 'srsue-sim-ue' chart] *****
changed: [node1]

TASK [helm/prepare : Compress custom Helm chart directory for 'srsue-sim-ue' if provided on localhost] ***
skipping: [node1]

TASK [helm/prepare : Compress built-in Helm chart directory for 'srsue-sim-ue' on localhost] ***
changed: [node1 -> localhost]

TASK [helm/prepare : Extract Helm chart archive for 'srsue-sim-ue' from localhost] ***
changed: [node1]

TASK [helm/prepare : Remove temporary Helm chart archive for 'srsue-sim-ue' from localhost] ***
changed: [node1 -> localhost]

TASK [helm/prepare : Render built-in Helm Chart.yaml templated file for 'srsue-sim-ue'] ***
changed: [node1]

TASK [helm/prepare : Render custom templated Helm files for 'srsue-sim-ue' if provided] ***
skipping: [node1]

TASK [helm/prepare : Render built-in templated Helm files for 'srsue-sim-ue'] ***
changed: [node1] => (item=val

Get SRSUE ZMQ UE logs:

In [14]:
print(await ue.get_logs(components=["sim_ue"], quiet=True))

Active RF plugins: libsrsran_rf_uhd.so libsrsran_rf_zmq.so
Inactive RF plugins: 
Reading configuration file /etc/config/ue.conf...

Built in Release mode using commit 6bcbd9e on branch HEAD.

2026-11-27T19:45:46.173080 [UE     ] [I] Using binary /usr/local/bin/srsue with arguments: /etc/config/ue.conf 
2026-11-27T19:45:46.224280 [UE     ] [I] Built in Release mode using commit 6bcbd9e on branch HEAD.
Opening 1 channels in RF device=zmq with args=tx_port=tcp://10.244.77.231:2001,rx_port=tcp://10.244.137.230:2000,base_srate=11.52e6
Supported RF device list: UHD zmq file
CHx base_srate=11.52e6
Current sample rate is 1.92 MHz with a base rate of 11.52 MHz (x6 decimation)
2026-11-27T19:45:46.286138 [RF     ] [I] Configuring physical DL channel 0 with band-pass filter (0.0, 0.0)
2026-11-27T19:45:46.286138 [RF     ] [I] Configuring physical UL channel 0 with band-pass filter (0.0, 0.0)
CH0 rx_port=tcp://10.244.137.230:2000
CH0 tx_port=tcp://10.244.77.231:2001
Current sample rate is 11.52 MHz 

Get DU logs after UE was connected:

In [15]:
print(await ran.get_logs(components=["du"], quiet=True))


--== OCUDU DU (commit 050a2bb) ==--

2026-11-27T19:44:43.791708 [GNB     ] [I] Built in Release mode using commit 050a2bb on branch HEAD
2026-11-27T19:44:43.792615 [CONFIG  ] [I] Input configuration (only non-default values): 
dryrun: false
gnb_id: 1
gnb_id_bit_length: 32
gnb_du_id: 1
log:
  filename: stdout
  all_level: info
  config_level: info
metrics:
  autostart_stdout_metrics: true
f1ap:
  addrs: 10.244.167.11
  bind_addrs: 10.244.137.230
f1u:
  bind_port: 2153
  peer_port: 2153
  socket: "bind_addr: 10.244.137.230"
cell_cfg:
  pci: 0
  dl_arfcn: 368500
  band: 3
  common_scs: 15
  channel_bandwidth_MHz: 10
  nof_antennas_ul: 1
  nof_antennas_dl: 1
  plmn: 00101
  tac: 1
  slicing: "sst: 1\nsd: 16777215"
  ssb:
    ssb_period: 10
  pdcch:
    common:
      coreset0_index: 6
      ss0_index: 0
    dedicated:
      dci_format_0_1_and_1_1: false
      ss2_type: common
  pdsch:
    mcs_table: qam64
  pusch:
    mcs_table: qam64
  prach:
    prach_config_index: 1
ru_sdr:
  srate: 11.

Verify that the UE got a PDU session:

In [16]:
print(await ue.run_command(command=["ip", "addr"], quiet=True))

1: lo: <LOOPBACK,UP,LOWER_UP> mtu 65536 qdisc noqueue state UNKNOWN group default qlen 1000
    link/loopback 00:00:00:00:00:00 brd 00:00:00:00:00:00
    inet 127.0.0.1/8 scope host lo
       valid_lft forever preferred_lft forever
    inet6 ::1/128 scope host 
       valid_lft forever preferred_lft forever
2: eth0@if12: <BROADCAST,MULTICAST,UP,LOWER_UP> mtu 9166 qdisc noqueue state UP group default qlen 1000
    link/ether 62:3b:0a:9f:ee:1f brd ff:ff:ff:ff:ff:ff link-netnsid 0
    inet 10.244.77.231/32 scope global eth0
       valid_lft forever preferred_lft forever
    inet6 fe80::603b:aff:fe9f:ee1f/64 scope link 
       valid_lft forever preferred_lft forever
3: tun_srsue: <POINTOPOINT,MULTICAST,NOARP,UP,LOWER_UP> mtu 1500 qdisc fq_codel state UNKNOWN group default qlen 500
    link/none 
    inet 11.1.1.2/24 scope global tun_srsue
       valid_lft forever preferred_lft forever



#### Test connectivity:

In [17]:
print(await ue.run_command(command=["apt", "update"], quiet=True))

Get:1 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Get:2 http://security.ubuntu.com/ubuntu noble-security/restricted amd64 Packages [2277 kB]
Get:3 http://security.ubuntu.com/ubuntu noble-security/multiverse amd64 Packages [55.6 kB]
Get:4 http://security.ubuntu.com/ubuntu noble-security/main amd64 Packages [1514 kB]
Get:5 http://security.ubuntu.com/ubuntu noble-security/universe amd64 Packages [1563 kB]
Get:6 http://archive.ubuntu.com/ubuntu noble InRelease [256 kB]
Get:7 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Get:8 http://archive.ubuntu.com/ubuntu noble-backports InRelease [126 kB]
Get:9 http://archive.ubuntu.com/ubuntu noble/multiverse amd64 Packages [331 kB]
Get:10 http://archive.ubuntu.com/ubuntu noble/restricted amd64 Packages [117 kB]
Get:11 http://archive.ubuntu.com/ubuntu noble/main amd64 Packages [1808 kB]
Get:12 http://archive.ubuntu.com/ubuntu noble/universe amd64 Packages [19.3 MB]
Get:13 http://archive.ubuntu.com/ubuntu nob

In [18]:
print(await ue.run_command(command=["apt", "install", "-y", "iputils-ping"], quiet=True))

Reading package lists...
Building dependency tree...
Reading state information...
The following NEW packages will be installed:
  iputils-ping
0 upgraded, 1 newly installed, 0 to remove and 53 not upgraded.
Need to get 44.6 kB of archives.
After this operation, 124 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 iputils-ping amd64 3:20240117-1ubuntu0.1 [44.6 kB]
Fetched 44.6 kB in 30s (1475 B/s)
Selecting previously unselected package iputils-ping.
(Reading database ... 35148 files and directories currently installed.)
Preparing to unpack .../iputils-ping_3%3a20240117-1ubuntu0.1_amd64.deb ...
Unpacking iputils-ping (3:20240117-1ubuntu0.1) ...
Setting up iputils-ping (3:20240117-1ubuntu0.1) ...



In [19]:
print(await ue.run_command(command=["ping", "11.1.1.1", "-c", "6", "-I", "tun_srsue"], quiet=True))

PING 11.1.1.1 (11.1.1.1) from 11.1.1.2 tun_srsue: 56(84) bytes of data.
64 bytes from 11.1.1.1: icmp_seq=1 ttl=64 time=164 ms
64 bytes from 11.1.1.1: icmp_seq=2 ttl=64 time=50.3 ms
64 bytes from 11.1.1.1: icmp_seq=3 ttl=64 time=24.6 ms
64 bytes from 11.1.1.1: icmp_seq=4 ttl=64 time=31.9 ms
64 bytes from 11.1.1.1: icmp_seq=5 ttl=64 time=23.4 ms
64 bytes from 11.1.1.1: icmp_seq=6 ttl=64 time=33.3 ms

--- 11.1.1.1 ping statistics ---
6 packets transmitted, 6 received, 0% packet loss, time 5005ms
rtt min/avg/max/mdev = 23.428/54.676/164.492/49.890 ms



In [20]:
print(await ue.run_command(command=["ping", "8.8.8.8", "-c", "6", "-I", "tun_srsue"], quiet=True))

PING 8.8.8.8 (8.8.8.8) from 11.1.1.2 tun_srsue: 56(84) bytes of data.
64 bytes from 8.8.8.8: icmp_seq=1 ttl=114 time=31.4 ms
64 bytes from 8.8.8.8: icmp_seq=2 ttl=114 time=24.6 ms
64 bytes from 8.8.8.8: icmp_seq=3 ttl=114 time=38.0 ms
64 bytes from 8.8.8.8: icmp_seq=4 ttl=114 time=32.3 ms
64 bytes from 8.8.8.8: icmp_seq=5 ttl=114 time=27.7 ms
64 bytes from 8.8.8.8: icmp_seq=6 ttl=114 time=40.0 ms

--- 8.8.8.8 ping statistics ---
6 packets transmitted, 6 received, 0% packet loss, time 5003ms
rtt min/avg/max/mdev = 24.560/32.345/40.039/5.403 ms



## Test another split: Change RAN split from E1-split to F1-split

We can change the OCUDU RAN's split from E1 to F1 while keeping the already deployed Open5GS core.

To do so, remove the deployed UE and RAN:

In [21]:
await ue.remove(quiet=True)
await ran.remove(quiet=True)

Remove existing OCUDU E1s-split RAN components:

In [22]:
ran.remove_component("cuup")
ran.remove_component("cucp")
ran.remove_component("du")

Add CU that will replace the previous CU-CP and CU-UP:

In [23]:
from metamodel_3gpp.models.ran import CUComponent, RANComponentType
from ifran.models.ran import DeployedRANComponent

ran.add_component(
    name="cu",
    component_meta=CUComponent(type=RANComponentType.CU),
    component_deployment=DeployedRANComponent.from_yaml(
"""
type: cu
implementation: ocudu
advanced_config:
  inactivity_timer: 7200
"""
    ),
)

Update and add the F1-split RAN DU from the previous one:

In [24]:
du_meta = metamodel.ran.components["du"]
du_deployemnt = deployment_model.ran.components["du"]

# update cu_cp and cu_up references in du component
du_meta.config.cu_cp = "cu"
du_meta.config.cu_up = "cu"

ran.add_component(
    name="du",
    component_meta=du_meta,
    component_deployment=du_deployemnt,
)

Deploy the OCUDU F1-split RAN:

In [25]:
await ran.deploy()


PLAY [node1] *******************************************************************

PLAY [node1] *******************************************************************

TASK [helm/prepare : Create Helm build directory for 'ocudu-du' chart] *********

TASK [helm/prepare : Create Helm build directory for 'ocudu-cu' chart] *********
changed: [node1]

TASK [helm/prepare : Compress custom Helm chart directory for 'ocudu-cu' if provided on localhost] ***
skipping: [node1]

TASK [helm/prepare : Compress built-in Helm chart directory for 'ocudu-cu' on localhost] ***
changed: [node1 -> localhost]

TASK [helm/prepare : Extract Helm chart archive for 'ocudu-cu' from localhost] ***
ok: [node1]

TASK [helm/prepare : Compress custom Helm chart directory for 'ocudu-du' if provided on localhost] ***
skipping: [node1]

TASK [helm/prepare : Compress built-in Helm chart directory for 'ocudu-du' on localhost] ***
changed: [node1 -> localhost]

TASK [helm/prepare : Extract Helm chart archive for 'ocudu-du' fro

Redeploy the SRSUE based on the changed RAN:

In [26]:
ue = UEManagement(ran)

In [27]:
await ue.deploy()


PLAY [node1] *******************************************************************

TASK [helm/prepare : Create Helm build directory for 'srsue-sim-ue' chart] *****
ok: [node1]

TASK [helm/prepare : Compress custom Helm chart directory for 'srsue-sim-ue' if provided on localhost] ***
skipping: [node1]

TASK [helm/prepare : Compress built-in Helm chart directory for 'srsue-sim-ue' on localhost] ***
changed: [node1 -> localhost]

TASK [helm/prepare : Extract Helm chart archive for 'srsue-sim-ue' from localhost] ***
ok: [node1]

TASK [helm/prepare : Remove temporary Helm chart archive for 'srsue-sim-ue' from localhost] ***
changed: [node1 -> localhost]

TASK [helm/prepare : Render built-in Helm Chart.yaml templated file for 'srsue-sim-ue'] ***
ok: [node1]

TASK [helm/prepare : Render custom templated Helm files for 'srsue-sim-ue' if provided] ***
skipping: [node1]

TASK [helm/prepare : Render built-in templated Helm files for 'srsue-sim-ue'] ***
changed: [node1] => (item=values.yaml)
chang

Get DU logs after UE was connected:

In [28]:
print(await ran.get_logs(components=["du"], quiet=True))


--== OCUDU DU (commit 050a2bb) ==--

2026-11-27T19:49:38.156244 [GNB     ] [I] Built in Release mode using commit 050a2bb on branch HEAD
2026-11-27T19:49:38.156974 [CONFIG  ] [I] Input configuration (only non-default values): 
dryrun: false
gnb_id: 1
gnb_id_bit_length: 32
gnb_du_id: 1
log:
  filename: stdout
  all_level: info
  config_level: info
metrics:
  autostart_stdout_metrics: true
f1ap:
  addrs: 10.244.28.59
  bind_addrs: 10.244.224.120
f1u:
  bind_port: 2153
  peer_port: 2153
  socket: "bind_addr: 10.244.224.120"
cell_cfg:
  pci: 0
  dl_arfcn: 368500
  band: 3
  common_scs: 15
  channel_bandwidth_MHz: 10
  nof_antennas_ul: 1
  nof_antennas_dl: 1
  plmn: 00101
  tac: 1
  slicing: "sst: 1\nsd: 16777215"
  ssb:
    ssb_period: 10
  pdcch:
    common:
      coreset0_index: 6
      ss0_index: 0
    dedicated:
      dci_format_0_1_and_1_1: false
      ss2_type: common
  pdsch:
    mcs_table: qam64
  pusch:
    mcs_table: qam64
  prach:
    prach_config_index: 1
ru_sdr:
  srate: 11.5

Verify that the UE got a PDU session:

In [29]:
print(await ue.run_command(command=["ip", "addr"], quiet=True))

1: lo: <LOOPBACK,UP,LOWER_UP> mtu 65536 qdisc noqueue state UNKNOWN group default qlen 1000
    link/loopback 00:00:00:00:00:00 brd 00:00:00:00:00:00
    inet 127.0.0.1/8 scope host lo
       valid_lft forever preferred_lft forever
    inet6 ::1/128 scope host 
       valid_lft forever preferred_lft forever
2: eth0@if14: <BROADCAST,MULTICAST,UP,LOWER_UP> mtu 9166 qdisc noqueue state UP group default qlen 1000
    link/ether 7e:de:18:95:66:fe brd ff:ff:ff:ff:ff:ff link-netnsid 0
    inet 10.244.140.110/32 scope global eth0
       valid_lft forever preferred_lft forever
    inet6 fe80::7cde:18ff:fe95:66fe/64 scope link 
       valid_lft forever preferred_lft forever
3: tun_srsue: <POINTOPOINT,MULTICAST,NOARP,UP,LOWER_UP> mtu 1500 qdisc fq_codel state UNKNOWN group default qlen 500
    link/none 
    inet 11.1.1.3/24 scope global tun_srsue
       valid_lft forever preferred_lft forever



#### Test connectivity:

In [30]:
print(await ue.run_command(command=["apt", "update"], quiet=True))

Get:1 http://archive.ubuntu.com/ubuntu noble InRelease [256 kB]
Get:2 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Get:3 http://archive.ubuntu.com/ubuntu noble-backports InRelease [126 kB]
Get:4 http://archive.ubuntu.com/ubuntu noble/main amd64 Packages [1808 kB]
Get:5 http://archive.ubuntu.com/ubuntu noble/multiverse amd64 Packages [331 kB]
Get:6 http://archive.ubuntu.com/ubuntu noble/universe amd64 Packages [19.3 MB]
Get:7 http://archive.ubuntu.com/ubuntu noble/restricted amd64 Packages [117 kB]
Get:8 http://archive.ubuntu.com/ubuntu noble-updates/universe amd64 Packages [2184 kB]
Get:9 http://archive.ubuntu.com/ubuntu noble-updates/multiverse amd64 Packages [61.7 kB]
Get:10 http://archive.ubuntu.com/ubuntu noble-updates/restricted amd64 Packages [2403 kB]
Get:11 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 Packages [1843 kB]
Get:12 http://archive.ubuntu.com/ubuntu noble-backports/main amd64 Packages [49.0 kB]
Get:13 http://archive.ubuntu.com/ubuntu 

In [31]:
print(await ue.run_command(command=["apt", "install", "-y", "iputils-ping"], quiet=True))

Reading package lists...
Building dependency tree...
Reading state information...
The following NEW packages will be installed:
  iputils-ping
0 upgraded, 1 newly installed, 0 to remove and 53 not upgraded.
Need to get 44.6 kB of archives.
After this operation, 124 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 iputils-ping amd64 3:20240117-1ubuntu0.1 [44.6 kB]
Fetched 44.6 kB in 0s (123 kB/s)
Selecting previously unselected package iputils-ping.
(Reading database ... 35148 files and directories currently installed.)
Preparing to unpack .../iputils-ping_3%3a20240117-1ubuntu0.1_amd64.deb ...
Unpacking iputils-ping (3:20240117-1ubuntu0.1) ...
Setting up iputils-ping (3:20240117-1ubuntu0.1) ...



In [32]:
print(await ue.run_command(command=["ping", "11.1.1.1", "-c", "6", "-I", "tun_srsue"], quiet=True))

PING 11.1.1.1 (11.1.1.1) from 11.1.1.3 tun_srsue: 56(84) bytes of data.
64 bytes from 11.1.1.1: icmp_seq=1 ttl=64 time=67.9 ms
64 bytes from 11.1.1.1: icmp_seq=2 ttl=64 time=62.0 ms
64 bytes from 11.1.1.1: icmp_seq=3 ttl=64 time=25.2 ms
64 bytes from 11.1.1.1: icmp_seq=4 ttl=64 time=39.4 ms
64 bytes from 11.1.1.1: icmp_seq=5 ttl=64 time=38.2 ms
64 bytes from 11.1.1.1: icmp_seq=6 ttl=64 time=34.5 ms

--- 11.1.1.1 ping statistics ---
6 packets transmitted, 6 received, 0% packet loss, time 5004ms
rtt min/avg/max/mdev = 25.189/44.508/67.876/15.239 ms



In [33]:
print(await ue.run_command(command=["ping", "8.8.8.8", "-c", "6", "-I", "tun_srsue"], quiet=True))

PING 8.8.8.8 (8.8.8.8) from 11.1.1.3 tun_srsue: 56(84) bytes of data.
64 bytes from 8.8.8.8: icmp_seq=1 ttl=114 time=32.9 ms
64 bytes from 8.8.8.8: icmp_seq=2 ttl=114 time=25.6 ms
64 bytes from 8.8.8.8: icmp_seq=3 ttl=114 time=39.6 ms
64 bytes from 8.8.8.8: icmp_seq=4 ttl=114 time=38.3 ms
64 bytes from 8.8.8.8: icmp_seq=5 ttl=114 time=30.7 ms
64 bytes from 8.8.8.8: icmp_seq=6 ttl=114 time=44.1 ms

--- 8.8.8.8 ping statistics ---
6 packets transmitted, 6 received, 0% packet loss, time 5007ms
rtt min/avg/max/mdev = 25.562/35.201/44.087/6.152 ms

